**bold text**### Tutorial 6 Transfer Learning on Rheumatoid Arthritis Proteomic Data
Transfer learning using PyTorch with AlexNet, ResNet101, and MobileNet architectures applied to NCBI GEO proteomic datasets.

In [ ]:
!pip install GEOparse biopython pandas torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.3 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
import GEOparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


### 1. Data Acquisition from NCBI GEO
Use a sample Rheumatoid Arthritis dataset (GSE15573).

In [ ]:
def load_geo_data(geo_id='GSE15573'):
    gse = GEOparse.get_GEO(geo_id, destdir='./')
    # Extract expression data
    data = gse.pivot_samples('VALUE')

    # Display columns to debug if necessary
    print("Available metadata columns:", gse.phenotype_data.columns)

    # Try to find a column containing 'characteristics' or similar
    char_cols = [c for c in gse.phenotype_data.columns if 'characteristics' in c.lower()]
    if char_cols:
        target_col = char_cols[0]
        labels = gse.phenotype_data[target_col].apply(lambda x: 1 if 'rheumatoid arthritis' in str(x).lower() else 0)
    else:
        # Fallback search across all metadata
        labels = gse.phenotype_data.apply(lambda row: 1 if row.astype(str).str.contains('rheumatoid arthritis', case=False).any() else 0, axis=1)

    return data.T, labels

X, y = load_geo_data()
print(f'Dataset shape: {X.shape}')
print(f'Label counts:\n{y.value_counts()}')

25-Apr-2026 14:52:19 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
25-Apr-2026 14:52:19 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
25-Apr-2026 14:52:19 INFO GEOparse - Parsing ./GSE15573_family.soft.gz: 
INFO:GEOparse:Parsing ./GSE15573_family.soft.gz: 
25-Apr-2026 14:52:19 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
25-Apr-2026 14:52:19 DEBUG GEOparse - SERIES: GSE15573
DEBUG:GEOparse:SERIES: GSE15573
25-Apr-2026 14:52:19 DEBUG GEOparse - PLATFORM: GPL6102
DEBUG:GEOparse:PLATFORM: GPL6102
25-Apr-2026 14:52:20 DEBUG GEOparse - SAMPLE: GSM389703
DEBUG:GEOparse:SAMPLE: GSM389703
25-Apr-2026 14:52:20 DEBUG GEOparse - SAMPLE: GSM389704
DEBUG:GEOparse:SAMPLE: GSM389704
25-Apr-2026 14:52:20 DEBUG GEOparse - SAMPLE: GSM389705
DEBUG:GEOparse:SAMPLE: GSM389705
25-Apr-2026 14:52:20 DEBUG GEOparse - SAMPLE: GSM389706
DEBUG:GEOparse:SAMPLE: GSM

Available metadata columns: Index(['title', 'geo_accession', 'status', 'submission_date',
       'last_update_date', 'type', 'channel_count', 'source_name_ch1',
       'organism_ch1', 'taxid_ch1', 'characteristics_ch1.0.status',
       'characteristics_ch1.1.gender', 'characteristics_ch1.2.age',
       'characteristics_ch1.3.tissue', 'molecule_ch1', 'extract_protocol_ch1',
       'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol',
       'description', 'data_processing', 'platform_id', 'contact_name',
       'contact_email', 'contact_phone', 'contact_fax', 'contact_laboratory',
       'contact_department', 'contact_institute', 'contact_address',
       'contact_city', 'contact_zip/postal_code', 'contact_country',
       'supplementary_file', 'series_id', 'data_row_count'],
      dtype='object')
Dataset shape: (33, 48701)
Label counts:
characteristics_ch1.0.status
1    18
0    15
Name: count, dtype: int64


### 2. Architecture Experimentation
Define a wrapper to adapt Pre-trained Computer Vision models for 1D Proteomic data by reshaping features into pseudo-images (224x224).

In [ ]:
def get_model(arch_name, num_classes=2):
    if arch_name == 'alexnet':
        model = models.alexnet(weights='IMAGENET1K_V1')
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif arch_name == 'resnet101':
        model = models.resnet101(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch_name == 'mobilenet':
        model = models.mobilenet_v2(weights='IMAGENET1K_V1')
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(device)

# Example: Initialize AlexNet
alex_net = get_model('alexnet')
print("AlexNet initialized for Transfer Learning.")

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:03<00:00, 72.2MB/s]


AlexNet initialized for Transfer Learning.
